In [2]:
import os
import glob
import json
import random
from typing import List, Tuple, Dict

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from affine import Affine

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, jaccard_score, classification_report
import joblib

In [3]:
# CONFIGURATION
CONFIG = {
    "safe_dir": r"C:\Users\ASUS\Desktop\Desertation\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE",
    "label_tif": r"C:\Users\ASUS\Desktop\Desertation\ESA_WorldCover_10m_2021_v200_N52W003_S2RGBNIR.tif",
    "out_dir": r"C:\Users\ASUS\Desktop\Desertation\Outputs/output_run1",

    # Sentinel-2 band patterns
    "bands_10m": [
        "*B02_10m.jp2",  # Blue
        "*B03_10m.jp2",  # Green
        "*B04_10m.jp2",  # Red
        "*B08_10m.jp2",  # NIR
    ],
    "bands_20m": [
        "*B11_20m.jp2",  # SWIR1
        "*B12_20m.jp2",  # SWIR2
    ],

    # Spatial downsampling factor (for memory safety)
    # 1.0  -> full 10m resolution (heavy)
    # 0.5  -> ~20m
    # 0.25 -> ~40m  (recommended if RAM is limited)
    "scale": 0.125,

    # Patching & training
    "patch_size": 128,
    "train_val_split": 0.8,
    "batch_size": 1,      # lower if you get CUDA OOM
    "num_epochs": 5,     # start small, you can increase later
    "learning_rate": 1e-3,

    # WorldCover classes (11, class 0 = unlabeled)
    "num_classes": 11,

    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Misc
    "seed": 42,
    "save_checkpoint_every": 5,
}

os.makedirs(CONFIG["out_dir"], exist_ok=True)
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

In [4]:
# UTILITIES – FIND BANDS
def find_band_files(safe_dir: str, patterns: List[str]) -> Dict[str, str]:

    matches = {}

    for pattern in patterns:
        # 1) Try the pattern as given
        search1 = os.path.join(safe_dir, "**", pattern)
        found = glob.glob(search1, recursive=True)

        # 2) Try without "_10m"/"_20m"
        if not found and "_10m" in pattern:
            alt = pattern.replace("_10m", "")
            search2 = os.path.join(safe_dir, "**", alt)
            found = glob.glob(search2, recursive=True)

        if not found and "_20m" in pattern:
            alt = pattern.replace("_20m", "")
            search2 = os.path.join(safe_dir, "**", alt)
            found = glob.glob(search2, recursive=True)

        # 3) Try generic *B0X*.jp2
        if not found and "B0" in pattern:
            band_id = pattern.split("B0")[-1].split("_")[0]  # e.g. "2" from "*B02_10m.jp2"
            generic = f"*B0{band_id}*.jp2"
            search3 = os.path.join(safe_dir, "**", generic)
            found = glob.glob(search3, recursive=True)

        if not found:
            # Debug: show what's actually inside the SAFE
            all_jp2 = glob.glob(os.path.join(safe_dir, "**", "*.jp2"), recursive=True)
            print(f"\n[DEBUG] No files found for pattern '{pattern}' in {safe_dir}")
            print(f"[DEBUG] Example JP2 files found in SAFE:")
            for p in all_jp2[:10]:
                print("   ", p)
            raise FileNotFoundError(f"No files found for pattern {pattern} in {safe_dir}")

        matches[pattern] = found[0]

    return matches

In [5]:
# BAND READING, RESAMPLING, STACKING
def read_band_as_array(path: str, scale: float = 0.25) -> Tuple[np.ndarray, dict]:

    with rasterio.open(path) as src:
        if scale == 1.0:
            arr = src.read(1).astype(np.float32)
            profile = src.profile.copy()
        else:
            new_height = int(src.height * scale)
            new_width = int(src.width * scale)

            arr = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.bilinear,
            ).astype(np.float32)

            scale_x = src.width / new_width
            scale_y = src.height / new_height
            new_transform = src.transform * Affine.scale(scale_x, scale_y)

            profile = src.profile.copy()
            profile.update(
                height=new_height,
                width=new_width,
                transform=new_transform,
            )

    return arr, profile


def resample_to_match(src_path: str, target_profile: dict) -> np.ndarray:

    with rasterio.open(src_path) as src:
        params = dict(
            crs=target_profile["crs"],
            transform=target_profile["transform"],
            width=target_profile["width"],
            height=target_profile["height"],
            resampling=Resampling.bilinear,
        )
        with WarpedVRT(src, **params) as vrt:
            arr = vrt.read(1).astype(np.float32)
    return arr


def stack_bands(b10_paths: List[str], b20_paths: List[str], out_path: str, scale: float) -> Tuple[np.ndarray, dict]:
s the in-memory array as float16 (half precision)

    band_arrays = []
    base_profile = None

    # 1) Load 10m bands (base geometry)
    for p in b10_paths:
        arr, prof = read_band_as_array(p, scale=scale)
        band_arrays.append(arr)
        if base_profile is None:
            base_profile = prof

    # 2) Load & resample 20m bands to the same grid
    for p in b20_paths:
        if p:
            arr = resample_to_match(p, base_profile)
            band_arrays.append(arr)

    # 3) Stack into single array (C, H, W) in float16 to save RAM
    stacked = np.stack(band_arrays, axis=0).astype(np.float16)
    del band_arrays  # free list memory ASAP

    # 4) Normalize in-place to [0,1]
    stacked /= 10000.0
    np.clip(stacked, 0.0, 1.0, out=stacked)

    # 5) Prepare metadata for GeoTIFF
    #    We still write as float32 on disk; GDAL handles the cast.
    meta = base_profile.copy()
    meta.update(
        driver="GTiff",
        count=stacked.shape[0],
        dtype="float32",   # file on disk
        compress="lzw",
    )

    # 6) Write GeoTIFF (will internally cast float16 -> float32 in chunks)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(stacked.astype(np.float32))

    return stacked, meta 

In [6]:
# LABEL ALIGNMENT
def load_label_and_align(label_tif: str, ref_meta: dict) -> Tuple[np.ndarray, dict]:
  
    with rasterio.open(label_tif) as src:
        params = dict(
            crs=ref_meta["crs"],
            transform=ref_meta["transform"],
            width=ref_meta["width"],
            height=ref_meta["height"],
            resampling=Resampling.nearest,
        )
        with WarpedVRT(src, **params) as vrt:
            label = vrt.read(1).astype(np.int16)
            out_profile = ref_meta.copy()
            out_profile.update(count=1, dtype="int16")
    return label, out_profile

In [7]:
# PATCHES & DATASET
def make_patches(
    image: np.ndarray,
    label: np.ndarray,
    patch_size: int = 256,
    stride: int = None,
    min_valid_fraction: float = 0.01,
) -> List[Tuple[np.ndarray, np.ndarray]]:

    C, H, W = image.shape
    if stride is None:
        stride = patch_size

    patches = []
    total = 0
    kept = 0

    for r in range(0, H - patch_size + 1, stride):
        for c in range(0, W - patch_size + 1, stride):
            total += 1
            img_patch = image[:, r:r+patch_size, c:c+patch_size]
            lbl_patch = label[r:r+patch_size, c:c+patch_size]

            valid_frac = np.count_nonzero(lbl_patch) / (patch_size * patch_size)
            if valid_frac >= min_valid_fraction:
                patches.append((img_patch, lbl_patch))
                kept += 1

    print(f"  [make_patches] total windows={total}, kept={kept}, min_valid_fraction={min_valid_fraction}")
    return patches


class PatchDataset(Dataset):


    def __init__(self, patches: List[Tuple[np.ndarray, np.ndarray]], augment: bool = False):
        self.patches = patches
        self.augment = augment

    def __len__(self) -> int:
        return len(self.patches)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img, lbl = self.patches[idx]

        # Simple augment: flips & rotations
        if self.augment:
            if random.random() > 0.5:
                img = np.flip(img, axis=2).copy()
                lbl = np.flip(lbl, axis=1).copy()
            if random.random() > 0.5:
                img = np.flip(img, axis=1).copy()
                lbl = np.flip(lbl, axis=0).copy()
            k = random.randint(0, 3)
            if k:
                img = np.rot90(img, k, axes=(1, 2)).copy()
                lbl = np.rot90(lbl, k).copy()

        img_t = torch.tensor(img, dtype=torch.float32)
        lbl_t = torch.tensor(lbl, dtype=torch.long)
        return img_t, lbl_t

In [8]:
# U-Net Model
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv_block(x)


class UNetSmall(nn.Module):
    """
    Extra-light U-Net: channels 16-32-64 instead of 64-128-256
    to reduce memory usage on CPU.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        # Encoder
        self.dconv_down1 = DoubleConv(in_channels, 16)
        self.pool1 = nn.MaxPool2d(2)

        self.dconv_down2 = DoubleConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = DoubleConv(32, 64)

        # Decoder
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dconv_up2 = DoubleConv(32 + 32, 32)  # 32 up + 32 skip

        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dconv_up1 = DoubleConv(16 + 16, 16)   # 16 up + 16 skip

        # Output
        self.conv_last = nn.Conv2d(16, out_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        c1 = self.dconv_down1(x)
        p1 = self.pool1(c1)

        c2 = self.dconv_down2(p1)
        p2 = self.pool2(c2)

        # Bridge
        b = self.bridge(p2)

        # Decoder
        u2 = self.up2(b)
        u2 = torch.cat([u2, c2], dim=1)
        c_up2 = self.dconv_up2(u2)

        u1 = self.up1(c_up2)
        u1 = torch.cat([u1, c1], dim=1)
        c_up1 = self.dconv_up1(u1)

        out = self.conv_last(c_up1)
        return out

In [9]:
# METRICS & TRAINING
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> dict:
    """
    Compute per-class IoU, mean IoU, and macro F1. Ignores class 0.
    """
    labels = list(range(1, num_classes + 1))

    per_class_iou = jaccard_score(
        y_true, y_pred, average=None, labels=labels, zero_division=0
    )
    mean_iou = float(np.nanmean(per_class_iou))
    f1_macro = f1_score(
        y_true, y_pred, average="macro", labels=labels, zero_division=0
    )

    return {
        "per_class_iou": per_class_iou.tolist(),
        "mean_iou": mean_iou,
        "f1_macro": f1_macro,
    }


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: dict) -> nn.Module:
    device = cfg["device"]
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    best_val_iou = -1.0
    best_model_path = os.path.join(cfg["out_dir"], "best_unet.pth")

    for epoch in range(cfg["num_epochs"]):
        model.train()
        running_loss = 0.0

        for imgs, masks in train_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation
        model.eval()
        preds_list = []
        gts_list = []

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                preds = outputs.argmax(dim=1).cpu().numpy()
                gts = masks.numpy()
                preds_list.append(preds.reshape(-1))
                gts_list.append(gts.reshape(-1))

        preds_flat = np.concatenate(preds_list)
        gts_flat = np.concatenate(gts_list)

        mask_valid = gts_flat > 0
        preds_flat = preds_flat[mask_valid]
        gts_flat = gts_flat[mask_valid]

        metrics = compute_metrics(gts_flat, preds_flat, cfg["num_classes"])
        mean_iou = metrics["mean_iou"]

        print(
            f"Epoch {epoch+1}/{cfg['num_epochs']}: "
            f"loss={running_loss:.4f}, mIoU={mean_iou:.4f}, F1={metrics['f1_macro']:.4f}"
        )

        if (epoch + 1) % cfg["save_checkpoint_every"] == 0:
            ckpt_path = os.path.join(
                cfg["out_dir"], f"unet_epoch{epoch+1}_miou{mean_iou:.4f}.pth"
            )
            torch.save(model.state_dict(), ckpt_path)
            print(f"  Saved checkpoint: {ckpt_path}")

        if mean_iou > best_val_iou:
            best_val_iou = mean_iou
            torch.save(model.state_dict(), best_model_path)
            print(f"  *** New best model saved: {best_model_path}")

    print(f"Best validation mIoU: {best_val_iou:.4f}")
    return model

In [10]:
# Random Forest Baseline
def train_random_forest(
    image: np.ndarray,
    label: np.ndarray,
    max_samples: int = 3000,
    n_estimators: int = 20,
) -> Tuple[RandomForestClassifier, str]:

    C, H, W = image.shape
    X = image.reshape(C, -1).T        # (H*W, C)
    y = label.flatten()               # (H*W,)

    # Use only labeled pixels (>0)
    mask = y > 0
    X_lab = X[mask]
    y_lab = y[mask]

    if len(y_lab) == 0:
        raise RuntimeError("No labeled pixels available for Random Forest.")

    # 🔹 Cap number of samples to avoid memory issues
    n = min(max_samples, len(y_lab))
    idx_train = np.random.choice(len(y_lab), size=n, replace=False)
    X_train = X_lab[idx_train]
    y_train = y_lab[idx_train]

    # 🔹 Very small RF to save memory
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=1,          # 1 thread → less RAM overhead
        random_state=42,
    )

    rf.fit(X_train, y_train)

    # Simple test on another random subset (or same if few samples)
    idx_test = np.random.choice(len(y_lab), size=n, replace=False)
    X_test = X_lab[idx_test]
    y_test = y_lab[idx_test]
    y_pred = rf.predict(X_test)

    report = classification_report(y_test, y_pred, zero_division=0)
    return rf, report

In [11]:
# FULL SCENE INFERENCE
def predict_whole_raster(
    model: nn.Module,
    stacked_arr: np.ndarray,
    patch_size: int,
    device: str,
) -> np.ndarray:
    C, H, W = stacked_arr.shape
    out = np.zeros((H, W), dtype=np.uint8)

    model = model.to(device)
    model.eval()

    step = patch_size

    with torch.no_grad():
        for r in range(0, H - patch_size + 1, step):
            for c in range(0, W - patch_size + 1, step):
                patch = stacked_arr[:, r:r+patch_size, c:c+patch_size]
                inp = torch.tensor(patch, dtype=torch.float32).unsqueeze(0).to(device)
                logits = model(inp)
                pred = logits.argmax(dim=1).cpu().numpy()[0]
                out[r:r+patch_size, c:c+patch_size] = pred

    return out

In [12]:
def remap_worldcover_labels(labels: np.ndarray) -> tuple[np.ndarray, int]:
    """
    Generic remap of label raster to a compact range [0..N]:

        0 stays 0 (unlabeled / ignore)
        All unique non-zero values -> 1..N (sorted order)

    Returns:
        remapped_labels (np.ndarray, int64)
        n_classes (int)  # number of distinct non-zero classes
    """
    unique_vals, counts = np.unique(labels, return_counts=True)
    nonzero_vals = unique_vals[unique_vals > 0]

    print("  [remap] unique non-zero label values:", nonzero_vals)

    if len(nonzero_vals) == 0:
        print("  [remap] WARNING: no non-zero labels found in this tile.")
        # everything will stay 0
        return np.zeros_like(labels, dtype=np.int64), 0

    # Map each distinct non-zero value to 1..N
    out = np.zeros_like(labels, dtype=np.int64)
    mapping = {}
    for new_idx, val in enumerate(nonzero_vals, start=1):
        out[labels == val] = new_idx
        mapping[val] = new_idx

    print("  [remap] mapping used (raw_value -> new_class):", mapping)
    n_classes = len(nonzero_vals)
    return out, n_classes


# MAIN PIPELINE
def main(cfg: dict):
    print("=== Sentinel-2 Land Cover Pipeline: START ===")

    os.makedirs(cfg["out_dir"], exist_ok=True)

    # 1) Locate band files
    print("[1/9] Locating Sentinel-2 bands...")
    b10 = find_band_files(cfg["safe_dir"], cfg["bands_10m"])
    b20 = find_band_files(cfg["safe_dir"], cfg["bands_20m"])

    print("  10m bands:")
    for pat, p in b10.items():
        print(f"    {pat} -> {p}")
    print("  20m bands:")
    for pat, p in b20.items():
        print(f"    {pat} -> {p}")

    # 2) Stack bands
    print("[2/9] Stacking bands...")
    stack_path = os.path.join(cfg["out_dir"], "sentinel2_stack.tif")
    stacked, meta = stack_bands(list(b10.values()), list(b20.values()), stack_path, cfg["scale"])
    print(f"  Stacked image: {stack_path}")
    print(f"  Shape: {stacked.shape}")  # (C, H, W)

    # 3) Align labels
    print("[3/9] Aligning labels...")
    labels, label_meta = load_label_and_align(cfg["label_tif"], meta)
    print(f"  Label shape (raw): {labels.shape}")

    C, H, W = stacked.shape
    assert labels.shape == (H, W), f"Shape mismatch: image {(H, W)} vs label {labels.shape}"

    labels, n_classes = remap_worldcover_labels(labels)

    if n_classes == 0:
        raise RuntimeError("Remapped labels have no non-zero classes. Check that label_tif overlaps Sentinel-2 tile.")

    CONFIG["num_classes"] = n_classes
    print(f"  [info] Number of classes after remap: {CONFIG['num_classes']}")

    total_pixels = labels.size
    labeled_pixels = np.count_nonzero(labels)
    labeled_ratio = labeled_pixels / total_pixels
    print(f"  Labeled pixels after remap: {labeled_pixels} / {total_pixels} ({labeled_ratio:.4%})")

    # 4) Generate patches
    print("[4/9] Generating patches...")
    patches = make_patches(
    stacked,
    labels,
    patch_size=cfg["patch_size"],
    stride=cfg["patch_size"],
    min_valid_fraction=0.01,
    )
    print(f"  Total patches: {len(patches)}")
    if len(patches) == 0:
        raise RuntimeError("No patches generated. Check labels/tile overlap.")

    max_patches = 200
    if len(patches) > max_patches:
        patches = random.sample(patches, max_patches)
        print(f"  [info] Subsampled patches to {len(patches)} for memory reasons.")

    # 5) Train/val split
    print("[5/9] Splitting train/val...")
    random.shuffle(patches)
    split_idx = int(len(patches) * cfg["train_val_split"])
    train_patches = patches[:split_idx]
    val_patches = patches[split_idx:]
    print(f"  Train patches: {len(train_patches)}")
    print(f"  Val   patches: {len(val_patches)}")

    train_ds = PatchDataset(train_patches, augment=True)
    val_ds = PatchDataset(val_patches, augment=False)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

    # 6) Train U-Net
    print("[6/9] Training U-Net...")
    model = UNetSmall(
        in_channels=stacked.shape[0],
        out_channels=cfg["num_classes"] + 1
    )
    model = train_model(model, train_loader, val_loader, cfg)


    # 7) Random Forest baseline
    print("[7/9] Training Random Forest baseline (lightweight)...")
    rf, rf_report = train_random_forest(
        stacked,
        labels,
        max_samples=3000,   # reduce to 1000 if RAM is still an issue
        n_estimators=20,    # small forest to save memory
    )

    rf_path = os.path.join(cfg["out_dir"], "rf_baseline.joblib")
    joblib.dump(rf, rf_path)
    print(f"  RF model saved: {rf_path}")

    rf_report_path = os.path.join(cfg["out_dir"], "rf_report.txt")
    with open(rf_report_path, "w") as f:
        f.write(rf_report)
    print(f"  RF report saved: {rf_report_path}")


    # 8) Full-scene inference
    print("[8/9] Running full-scene inference...")
    pred_raster = predict_whole_raster(model, stacked, cfg["patch_size"], cfg["device"])
    pred_meta = meta.copy()
    pred_meta.update(driver="GTiff", count=1, dtype="uint8")
    pred_path = os.path.join(cfg["out_dir"], "pred_unet.tif")
    with rasterio.open(pred_path, "w", **pred_meta) as dst:
        dst.write(pred_raster.astype("uint8"), 1)
    print(f"  Prediction saved: {pred_path}")

    # 9) Global metrics
    print("[9/9] Computing global metrics...")
    mask_valid = labels > 0
    y_true = labels[mask_valid].flatten()
    y_pred = pred_raster[mask_valid].flatten()
    metrics = compute_metrics(y_true, y_pred, cfg["num_classes"])
    print("  Metrics:")
    print(json.dumps(metrics, indent=2))
    metrics_path = os.path.join(cfg["out_dir"], "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"  Metrics saved: {metrics_path}")

    print("=== Sentinel-2 Land Cover Pipeline: COMPLETED ===")


if __name__ == "__main__":
    main(CONFIG)

=== Sentinel-2 Land Cover Pipeline: START ===
[1/9] Locating Sentinel-2 bands...
  10m bands:
    *B02_10m.jp2 -> C:\Users\ASUS\Desktop\Desertation\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\GRANULE\L2A_T30UWD_A054503_20251128T111434\IMG_DATA\R10m\T30UWD_20251128T111431_B02_10m.jp2
    *B03_10m.jp2 -> C:\Users\ASUS\Desktop\Desertation\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\GRANULE\L2A_T30UWD_A054503_20251128T111434\IMG_DATA\R10m\T30UWD_20251128T111431_B03_10m.jp2
    *B04_10m.jp2 -> C:\Users\ASUS\Desktop\Desertation\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\S2A_MSIL2A_20251128T111431_N0511_R137_T30UWD_20251128T132409.SAFE\GRANULE\L2A_T30UWD_A054503_20251128T111434\IMG_DATA\R10m\T30UWD_20251128T111431_B04_10m.jp2
    *B08_10m.jp2 -> C:\Users\ASUS\Desktop\Desertation\S2A_MSIL2A_2025112

In [13]:
import os
import numpy as np
import torch
import rasterio
from torch.utils.data import DataLoader

device = CONFIG["device"]

# Load stacked Sentinel-2 image from the saved GeoTIFF ---
stack_path = os.path.join(CONFIG["out_dir"], "sentinel2_stack.tif")
with rasterio.open(stack_path) as src:
    stacked = src.read().astype(np.float32)  # (C, H, W)
    meta = src.profile

print("Loaded stacked image:", stacked.shape)

# Load & align labels, then remap to 0..num_classes ---
labels_raw, _ = load_label_and_align(CONFIG["label_tif"], meta)
labels, n_classes = remap_worldcover_labels(labels_raw)
print("Labels shape:", labels.shape)
print("Number of classes (non-zero):", n_classes)

CONFIG["num_classes"] = n_classes  # ensure consistency

# Load trained U-Net weights ---
model = UNetSmall(
    in_channels=stacked.shape[0],
    out_channels=CONFIG["num_classes"] + 1  # 0..K, with 0 ignored
)

# choose the best model or final one:
best_model_path = os.path.join(CONFIG["out_dir"], "best_unet.pth")
final_model_path = os.path.join(CONFIG["out_dir"], "unet_final.pth")

model_path_to_use = best_model_path if os.path.exists(best_model_path) else final_model_path
print("Loading model from:", model_path_to_use)

state_dict = torch.load(model_path_to_use, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()


Loaded stacked image: (6, 1372, 1372)
  [remap] unique non-zero label values: [  74   79   80 ... 7952 8259 8293]
  [remap] mapping used (raw_value -> new_class): {np.int16(74): 1, np.int16(79): 2, np.int16(80): 3, np.int16(81): 4, np.int16(83): 5, np.int16(84): 6, np.int16(85): 7, np.int16(88): 8, np.int16(92): 9, np.int16(93): 10, np.int16(94): 11, np.int16(98): 12, np.int16(99): 13, np.int16(102): 14, np.int16(104): 15, np.int16(105): 16, np.int16(106): 17, np.int16(107): 18, np.int16(108): 19, np.int16(109): 20, np.int16(110): 21, np.int16(111): 22, np.int16(112): 23, np.int16(114): 24, np.int16(116): 25, np.int16(117): 26, np.int16(118): 27, np.int16(119): 28, np.int16(120): 29, np.int16(121): 30, np.int16(123): 31, np.int16(124): 32, np.int16(126): 33, np.int16(127): 34, np.int16(128): 35, np.int16(129): 36, np.int16(130): 37, np.int16(131): 38, np.int16(132): 39, np.int16(133): 40, np.int16(134): 41, np.int16(135): 42, np.int16(136): 43, np.int16(137): 44, np.int16(138): 45, np.

UNetSmall(
  (dconv_down1): DoubleConv(
    (conv_block): Sequential(
      (0): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dconv_down2): DoubleConv(
    (conv_block): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReL

In [14]:
# Predict full scene (if not already done) ---
print("Running full-scene prediction for evaluation...")

pred_raster = predict_whole_raster(
    model,
    stacked,
    patch_size=CONFIG["patch_size"],
    device=device,
)

# save prediction for reference (if not already saved)
pred_meta = meta.copy()
pred_meta.update(driver="GTiff", count=1, dtype="uint8")
pred_path = os.path.join(CONFIG["out_dir"], "pred_unet_eval.tif")
with rasterio.open(pred_path, "w", **pred_meta) as dst:
    dst.write(pred_raster.astype("uint8"), 1)

print("Prediction saved to:", pred_path)

# Compute global metrics over valid pixels ---
mask_valid = labels > 0
y_true = labels[mask_valid].flatten()
y_pred = pred_raster[mask_valid].flatten()

# overall pixel accuracy
overall_acc = float((y_true == y_pred).mean())
print(f"Overall pixel accuracy: {overall_acc:.4f}")

# IoU + F1 using your existing compute_metrics
global_metrics = compute_metrics(y_true, y_pred, CONFIG["num_classes"])
print("Global metrics (IoU, F1):")
import json
print(json.dumps(global_metrics, indent=2))

# Save metrics for your dissertation
metrics_path = os.path.join(CONFIG["out_dir"], "metrics_full_scene.json")
out_obj = {
    "overall_pixel_accuracy": overall_acc,
    **global_metrics,
}
with open(metrics_path, "w") as f:
    json.dump(out_obj, f, indent=2)

print("Saved full-scene metrics to:", metrics_path)

Running full-scene prediction for evaluation...
Prediction saved to: C:\Users\ASUS\Desktop\Desertation\Outputs/output_run1\pred_unet_eval.tif
Overall pixel accuracy: 0.0005
Global metrics (IoU, F1):
{
  "per_class_iou": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    

In [15]:
# Create patches again, but ONLY for evaluation (no training) ---
print("Creating patches for testing/evaluation...")

eval_patches = make_patches(
    stacked,
    labels,
    patch_size=CONFIG["patch_size"],
    stride=CONFIG["patch_size"],
    min_valid_fraction=0.01,   # require at least 1% labeled pixels
)

print("Total patches available:", len(eval_patches))

# To avoid memory issues, subsample a limited number of patches
max_eval_patches = 100  # adjust if you want more/less
if len(eval_patches) > max_eval_patches:
    import random
    eval_patches = random.sample(eval_patches, max_eval_patches)
    print(f"Subsampled patches to {len(eval_patches)} for evaluation.")

eval_ds = PatchDataset(eval_patches, augment=False)
eval_loader = DataLoader(eval_ds, batch_size=1, shuffle=False, num_workers=0)

criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

total_loss = 0.0
total_pixels = 0
correct_pixels = 0

with torch.no_grad():
    for imgs, masks in eval_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss = criterion(logits, masks)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        # compute accuracy on labeled pixels only
        mask_valid = masks > 0
        correct_pixels += (preds[mask_valid] == masks[mask_valid]).sum().item()
        total_pixels += mask_valid.sum().item()

avg_test_loss = total_loss / max(1, len(eval_loader))
test_accuracy = correct_pixels / max(1, total_pixels)

print(f"Average test loss (cross-entropy): {avg_test_loss:.4f}")
print(f"Test pixel accuracy (on labeled pixels): {test_accuracy:.4f}")

# Save to JSON file
test_metrics_path = os.path.join(CONFIG["out_dir"], "metrics_test_patches.json")
test_obj = {
    "avg_test_loss": float(avg_test_loss),
    "test_pixel_accuracy": float(test_accuracy),
    "num_eval_patches": len(eval_patches),
}
with open(test_metrics_path, "w") as f:
    json.dump(test_obj, f, indent=2)

print("Saved test patch metrics to:", test_metrics_path)

Creating patches for testing/evaluation...
  [make_patches] total windows=100, kept=56, min_valid_fraction=0.01
Total patches available: 56
Average test loss (cross-entropy): 6.9039
Test pixel accuracy (on labeled pixels): 0.0078
Saved test patch metrics to: C:\Users\ASUS\Desktop\Desertation\Outputs/output_run1\metrics_test_patches.json
